# Six non-protein quality indicator regression

This notebook runs regression for six non-protein quality indicators using the same fixed 113 training samples and 36 independent validation samples exported by `Protein_regression_GitHub_ready.ipynb`.

Run the revised protein notebook first. This notebook loads the exact split ID files and the fixed combined 149-sample quality table used in the original analysis. Model definitions, parameters, preprocessing, GroupKFold workflow, sample-level aggregation, and performance calculations are unchanged.

In [ ]:
# ============================================================
# Six non-protein quality indicator regression
# Same fixed 113 training samples and 36 validation samples as protein regression
# No repeated validation
# ============================================================

import os
import re
import json
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, ElasticNet

import joblib


# ============================================================
# 0. Settings
# ============================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root(start_path=None):
    """
    Find the repository root by searching the current directory and its
    parents for the required 'Pea samples-new' data directory.
    """
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "Pea samples-new").is_dir():
            return candidate

    raise FileNotFoundError(
        "Cannot locate the project root. Run this notebook from inside the "
        "GitHub repository and make sure the repository contains the "
        "'Pea samples-new' directory."
    )


PROJECT_DIR = find_project_root()
DATA_BASE = PROJECT_DIR / "Pea samples-new"
MODEL_DIR = DATA_BASE / "Regression model"

# Fixed input paths. These match the data sources used in the original run.
A_PATCH_CSV = DATA_BASE / "pea_patch_dataset.csv"
B_PATCH_CSV = MODEL_DIR / "flour_external_pea_patch_dataset.csv"
B_QUALITY_XLSX = MODEL_DIR / "pea quality-2025.xlsx"

# Fixed combined quality table used by the original six-indicator run.
QUALITY_ALL_CSV = (
    PROJECT_DIR
    / "Pea_TwoSource_Mixed113_Model"
    / "combined_A_B_quality_all149.csv"
)

# Exact split-ID files exported by the revised protein regression notebook.
PROTEIN_MODEL_DIR = PROJECT_DIR / "Pea_TwoSource_Mixed113_ProteinRegression_Model"
TRAIN_ID_CSV = PROTEIN_MODEL_DIR / "fixed_training_sample_ids.csv"
VAL_ID_CSV = PROTEIN_MODEL_DIR / "fixed_validation_sample_ids.csv"

required_input_files = [
    A_PATCH_CSV,
    B_PATCH_CSV,
    QUALITY_ALL_CSV,
    TRAIN_ID_CSV,
    VAL_ID_CSV,
]

missing_input_files = [path for path in required_input_files if not path.is_file()]
if missing_input_files:
    missing_text = "\n".join(f"- {path}" for path in missing_input_files)
    raise FileNotFoundError(
        "The following required input files were not found:\n"
        f"{missing_text}\n\n"
        "Run Protein_regression_GitHub_ready.ipynb first so that the fixed "
        "training and validation ID files are created."
    )

OUT_ROOT = PROJECT_DIR / "Pea_TwoSource_Mixed113_SixIndicatorRegression_SameSplitAsProtein"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_COLS = [
    "Moisture content",
    "Water uptaking capacity",
    "Water binding capacity",
    "Oil binding capacity",
    "Protein solubility",
    "Peak viscosity (RVA)",
]

QUALITY_COLS_ALL = [
    "Protein content",
    "Moisture content",
    "Water uptaking capacity",
    "Water binding capacity",
    "Oil binding capacity",
    "Protein solubility",
    "Peak viscosity (RVA)",
]

MODEL_NAMES_TO_RUN = [
    "PLSR_10",
    "PLSR_15",
    "SVR_RBF",
    "Ridge",
    "ElasticNet",
]

DROP_MISSING_TARGET_IN_SAME_SPLIT = False

# Model definitions and parameters are intentionally unchanged to preserve results.

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 20
plt.rcParams["axes.labelsize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["xtick.labelsize"] = 20
plt.rcParams["ytick.labelsize"] = 20
plt.rcParams["legend.fontsize"] = 20


# ============================================================
# 1. Basic helper functions
# ============================================================


def clean_sample_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s


def safe_name(s):
    s = str(s)
    s = s.replace("%", "pct")
    s = re.sub(r"[^\w\-]+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_")


def detect_sample_id_column(df):
    candidates = [
        "sample_id", "Sample ID", "Sample_ID", "Sample", "sample", "ID", "id"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        "Cannot detect sample id column. Available columns:\n"
        f"{list(df.columns)}"
    )


def detect_wavelength_columns(df):
    """
    Detect spectral wavelength columns.
    Supports: 997.71, 997.71nm, 997.71 nm, X997.71, wavelength_997.71, band_997.71.
    """
    wave_cols = []
    wave_values = []

    for c in df.columns:
        c_str = str(c).strip()

        try:
            w = float(c_str)
            if 900 <= w <= 2600:
                wave_cols.append(c)
                wave_values.append(w)
                continue
        except Exception:
            pass

        nums = re.findall(r"\d+\.\d+|\d+", c_str)
        if len(nums) > 0:
            try:
                w = float(nums[0])
                if 900 <= w <= 2600:
                    wave_cols.append(c)
                    wave_values.append(w)
            except Exception:
                pass

    if len(wave_cols) == 0:
        raise ValueError(
            "No wavelength columns detected.\n\n"
            "First 30 columns:\n"
            f"{list(df.columns[:30])}\n\n"
            "Last 30 columns:\n"
            f"{list(df.columns[-30:])}"
        )

    order = np.argsort(wave_values)
    wave_cols = [wave_cols[i] for i in order]
    wave_values = np.array([wave_values[i] for i in order], dtype=float)

    print(f"Detected {len(wave_cols)} wavelength columns.")
    print("First 5 wavelengths:", wave_values[:5])
    print("Last 5 wavelengths:", wave_values[-5:])

    return wave_cols, wave_values


def normalize_quality_column_names(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    rename_map = {}

    for c in df.columns:
        c_low = c.lower().strip()

        if c_low in ["protein", "protein content", "protein_content"]:
            rename_map[c] = "Protein content"
        elif c_low in ["moisture", "moisture content", "moisture_content"]:
            rename_map[c] = "Moisture content"
        elif c_low in [
            "water uptaking capacity", "water uptake capacity", "wuc",
            "water_uptaking_capacity", "water_uptake_capacity"
        ]:
            rename_map[c] = "Water uptaking capacity"
        elif c_low in ["water binding capacity", "wbc", "water_binding_capacity"]:
            rename_map[c] = "Water binding capacity"
        elif c_low in ["oil binding capacity", "obc", "oil_binding_capacity"]:
            rename_map[c] = "Oil binding capacity"
        elif c_low in ["protein solubility", "solubility", "protein_solubility"]:
            rename_map[c] = "Protein solubility"
        elif c_low in [
            "peak viscosity", "peak viscosity (rva)", "rva", "peak_viscosity", "peak_viscosity_rva"
        ]:
            rename_map[c] = "Peak viscosity (RVA)"

    return df.rename(columns=rename_map)


def add_b_treatment_from_sample_id(df):
    df = df.copy()

    def map_treatment(sid):
        try:
            sid_int = int(float(str(sid)))
        except Exception:
            return "Unknown"
        if 1 <= sid_int <= 12:
            return "Control"
        if 13 <= sid_int <= 24:
            return "150% N"
        if 25 <= sid_int <= 36:
            return "50% water"
        return "Unknown"

    df["treatment"] = df["sample_id"].apply(map_treatment)
    return df


# ============================================================
# 2. Data loading functions
# ============================================================

def prepare_patch_table(patch_csv, source_name):
    patch_df = pd.read_csv(patch_csv)
    patch_df.columns = [str(c).strip() for c in patch_df.columns]

    sample_col = detect_sample_id_column(patch_df)
    patch_df = patch_df.rename(columns={sample_col: "sample_id"})
    patch_df["sample_id"] = patch_df["sample_id"].apply(clean_sample_id)

    patch_df["source"] = source_name
    patch_df["global_sample_id"] = np.where(
        source_name == "A_original",
        "A_" + patch_df["sample_id"].astype(str),
        "B_" + patch_df["sample_id"].astype(str),
    )

    if source_name == "B_second":
        patch_df = add_b_treatment_from_sample_id(patch_df)
    else:
        patch_df["treatment"] = "Original"

    wave_cols, wavelengths = detect_wavelength_columns(patch_df)
    return patch_df, wave_cols, wavelengths


def build_quality_all_from_patch_and_excel(a_patch_df, b_quality_xlsx):
    a_quality = normalize_quality_column_names(a_patch_df.copy())
    missing_a_quality = [c for c in QUALITY_COLS_ALL if c not in a_quality.columns]
    if len(missing_a_quality) > 0:
        raise ValueError(
            "A patch CSV does not contain all quality columns, and combined quality CSV was not found.\n"
            f"Missing A quality columns: {missing_a_quality}"
        )

    a_quality = (
        a_quality[["global_sample_id", "source", "sample_id", "treatment"] + QUALITY_COLS_ALL]
        .drop_duplicates(subset=["global_sample_id"])
        .reset_index(drop=True)
    )

    b_quality = pd.read_excel(b_quality_xlsx)
    b_quality = normalize_quality_column_names(b_quality)

    sample_col = detect_sample_id_column(b_quality)
    b_quality = b_quality.rename(columns={sample_col: "sample_id"})
    b_quality["sample_id"] = b_quality["sample_id"].apply(clean_sample_id)
    b_quality["source"] = "B_second"
    b_quality["global_sample_id"] = "B_" + b_quality["sample_id"].astype(str)

    if "treatment" not in b_quality.columns:
        b_quality = add_b_treatment_from_sample_id(b_quality)

    missing_b_quality = [c for c in QUALITY_COLS_ALL if c not in b_quality.columns]
    if len(missing_b_quality) > 0:
        raise ValueError(
            f"B quality Excel is missing quality columns: {missing_b_quality}\n"
            f"Available columns: {list(b_quality.columns)}"
        )

    b_quality = (
        b_quality[["global_sample_id", "source", "sample_id", "treatment"] + QUALITY_COLS_ALL]
        .drop_duplicates(subset=["global_sample_id"])
        .reset_index(drop=True)
    )

    return pd.concat([a_quality, b_quality], axis=0, ignore_index=True)


def load_or_build_quality_all(a_patch_df=None):
    """
    Load the exact combined 149-sample quality table used in the original run.

    The argument is retained for compatibility with the original function call.
    No alternative combined table or fallback reconstruction is used.
    """
    print("Using fixed combined quality table:")
    print(QUALITY_ALL_CSV)

    quality_all = pd.read_csv(QUALITY_ALL_CSV)
    quality_all = normalize_quality_column_names(quality_all)

    if "sample_id" not in quality_all.columns:
        sample_col = detect_sample_id_column(quality_all)
        quality_all = quality_all.rename(columns={sample_col: "sample_id"})
    quality_all["sample_id"] = quality_all["sample_id"].apply(clean_sample_id)

    if "source" not in quality_all.columns:
        raise ValueError("Combined quality table must contain column: source")

    if "global_sample_id" not in quality_all.columns:
        quality_all["global_sample_id"] = np.where(
            quality_all["source"].astype(str).str.contains("A", case=False, na=False),
            "A_" + quality_all["sample_id"].astype(str),
            "B_" + quality_all["sample_id"].astype(str),
        )

    if "treatment" not in quality_all.columns:
        quality_all["treatment"] = np.where(
            quality_all["source"].astype(str).str.contains("B", case=False, na=False),
            "Unknown",
            "Original",
        )
        b_mask = quality_all["source"].astype(str).str.contains("B", case=False, na=False)
        temp = add_b_treatment_from_sample_id(quality_all.loc[b_mask].copy())
        quality_all.loc[b_mask, "treatment"] = temp["treatment"].values

    quality_all["source"] = quality_all["source"].astype(str)
    quality_all.loc[
        quality_all["source"].str.contains("A", case=False, na=False), "source"
    ] = "A_original"
    quality_all.loc[
        quality_all["source"].str.contains("B", case=False, na=False), "source"
    ] = "B_second"

    quality_all["global_sample_id"] = quality_all["global_sample_id"].astype(str)
    quality_all["sample_id"] = quality_all["sample_id"].apply(clean_sample_id)

    b_mask = quality_all["source"] == "B_second"
    temp_b = add_b_treatment_from_sample_id(quality_all.loc[b_mask].copy())
    quality_all.loc[b_mask, "treatment"] = temp_b["treatment"].values
    quality_all.loc[quality_all["source"] == "A_original", "treatment"] = "Original"

    missing_quality_cols = [c for c in QUALITY_COLS_ALL if c not in quality_all.columns]
    if missing_quality_cols:
        raise ValueError(
            "Combined quality table is missing required columns: "
            f"{missing_quality_cols}"
        )

    for c in QUALITY_COLS_ALL:
        quality_all[c] = pd.to_numeric(quality_all[c], errors="coerce")

    keep_cols = [
        "global_sample_id",
        "source",
        "sample_id",
        "treatment",
    ] + QUALITY_COLS_ALL

    quality_all = (
        quality_all[keep_cols]
        .drop_duplicates(subset=["global_sample_id"])
        .reset_index(drop=True)
    )

    if quality_all["global_sample_id"].nunique() != 149:
        raise ValueError(
            "The combined quality table must contain 149 unique samples, but "
            f"found {quality_all['global_sample_id'].nunique()}."
        )

    return quality_all

# ============================================================
# 3. Protein split loading
# ============================================================

def read_global_sample_ids_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    if "global_sample_id" in df.columns:
        ids = df["global_sample_id"].astype(str).tolist()
    else:
        ids = df.iloc[:, 0].astype(str).tolist()
    return sorted(list(pd.unique(ids)))


def load_protein_split_ids():
    """
    Load the exact 113/36 split exported by the revised protein notebook.
    No recursive search or split reconstruction is used.
    """
    print("\nLoading fixed protein split IDs:")
    print(TRAIN_ID_CSV)
    print(VAL_ID_CSV)

    train_ids = read_global_sample_ids_from_csv(TRAIN_ID_CSV)
    val_ids = read_global_sample_ids_from_csv(VAL_ID_CSV)

    if len(train_ids) != 113:
        raise ValueError(
            f"Protein training split should have 113 samples, but got {len(train_ids)}."
        )
    if len(val_ids) != 36:
        raise ValueError(
            f"Protein validation split should have 36 samples, but got {len(val_ids)}."
        )

    overlap = set(train_ids).intersection(set(val_ids))
    if overlap:
        raise ValueError(f"Training and validation split overlap detected: {overlap}")

    quality_ids = set(quality_all["global_sample_id"].astype(str))
    missing_from_quality = sorted((set(train_ids) | set(val_ids)) - quality_ids)
    if missing_from_quality:
        raise ValueError(
            "Split IDs are missing from the combined quality table: "
            f"{missing_from_quality}"
        )

    split_dir = OUT_ROOT / "protein_split_ids_used"
    split_dir.mkdir(parents=True, exist_ok=True)

    pd.DataFrame({"global_sample_id": train_ids}).to_csv(
        split_dir / "fixed_training_sample_ids_used.csv",
        index=False,
    )
    pd.DataFrame({"global_sample_id": val_ids}).to_csv(
        split_dir / "fixed_validation_sample_ids_used.csv",
        index=False,
    )

    print("\nFixed protein split loaded successfully.")
    print("Training samples:", len(train_ids))
    print("Validation samples:", len(val_ids))

    print("\nTraining source counts:")
    display(
        quality_all[
            quality_all["global_sample_id"].astype(str).isin(train_ids)
        ]["source"].value_counts()
    )

    print("\nValidation source counts:")
    display(
        quality_all[
            quality_all["global_sample_id"].astype(str).isin(val_ids)
        ]["source"].value_counts()
    )

    print("\nValidation B treatment counts:")
    display(
        quality_all[
            quality_all["global_sample_id"].astype(str).isin(val_ids)
            & (quality_all["source"] == "B_second")
        ]["treatment"].value_counts()
    )

    print("\nSaved split IDs used to:")
    print(split_dir)

    return train_ids, val_ids


# ============================================================
# 4. Model functions
# ============================================================

class SNVTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        mean = np.nanmean(X, axis=1, keepdims=True)
        std = np.nanstd(X, axis=1, keepdims=True)
        std[std == 0] = 1.0
        return (X - mean) / std


def model_factory(model_name):
    if model_name == "PLSR_10":
        reg = PLSRegression(n_components=10)
    elif model_name == "PLSR_15":
        reg = PLSRegression(n_components=15)
    elif model_name == "SVR_RBF":
        reg = SVR(kernel="rbf", C=10, gamma="scale", epsilon=0.1)
    elif model_name == "Ridge":
        reg = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    elif model_name == "ElasticNet":
        reg = ElasticNet(alpha=0.01, l1_ratio=0.2, random_state=RANDOM_STATE, max_iter=10000)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("reg", reg),
    ])


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    rpd = np.std(y_true, ddof=1) / rmse if rmse != 0 and len(y_true) > 1 else np.nan

    return {"r2": r2, "rmse": rmse, "mae": mae, "bias": bias, "rpd": rpd}


def aggregate_patch_predictions(pred_patch_df):
    sample_pred = (
        pred_patch_df
        .groupby(["global_sample_id", "source", "sample_id", "treatment"], as_index=False)
        .agg(y_true=("y_true", "first"), y_pred=("y_pred", "mean"), n_patches=("y_pred", "size"))
    )
    sample_pred["residual"] = sample_pred["y_pred"] - sample_pred["y_true"]
    sample_pred["abs_error"] = sample_pred["residual"].abs()
    return sample_pred


def fit_predict_sample_level(model, train_patch_df, test_patch_df, wave_cols):
    X_train = train_patch_df[wave_cols].values.astype(float)
    y_train = train_patch_df["y_true"].values.astype(float)
    X_test = test_patch_df[wave_cols].values.astype(float)

    model.fit(X_train, y_train)
    y_pred_patch = np.asarray(model.predict(X_test)).reshape(-1)

    pred_patch_df = test_patch_df[["global_sample_id", "source", "sample_id", "treatment", "y_true"]].copy()
    pred_patch_df["y_pred"] = y_pred_patch

    sample_pred = aggregate_patch_predictions(pred_patch_df)
    return model, sample_pred


def groupkfold_oof_prediction(patch_train, wave_cols, model_name, n_splits=5):
    groups = patch_train["global_sample_id"].values
    unique_groups = patch_train["global_sample_id"].nunique()
    n_splits = min(n_splits, unique_groups)
    gkf = GroupKFold(n_splits=n_splits)

    all_fold_predictions = []
    for fold_id, (idx_tr, idx_te) in enumerate(gkf.split(patch_train, patch_train["y_true"], groups)):
        fold_train = patch_train.iloc[idx_tr].copy()
        fold_test = patch_train.iloc[idx_te].copy()
        model = model_factory(model_name)
        _, fold_sample_pred = fit_predict_sample_level(
            model=model,
            train_patch_df=fold_train,
            test_patch_df=fold_test,
            wave_cols=wave_cols,
        )
        fold_sample_pred["fold"] = fold_id
        all_fold_predictions.append(fold_sample_pred)

    oof_pred = pd.concat(all_fold_predictions, axis=0, ignore_index=True)
    metrics = regression_metrics(oof_pred["y_true"], oof_pred["y_pred"])
    metrics["model"] = model_name
    metrics["n_samples"] = oof_pred["global_sample_id"].nunique()
    metrics["n_patches"] = patch_train.shape[0]
    metrics["cv"] = f"GroupKFold_{n_splits}"
    return metrics, oof_pred


def prepare_patch_for_target(patch_all, quality_all, target_col):
    quality_target = quality_all[["global_sample_id", "source", "sample_id", "treatment", target_col]].copy()
    quality_target = quality_target.rename(columns={target_col: "y_true"})
    quality_target["y_true"] = pd.to_numeric(quality_target["y_true"], errors="coerce")

    patch_target = patch_all.merge(
        quality_target[["global_sample_id", "y_true"]],
        on="global_sample_id",
        how="inner",
    )
    return patch_target, quality_target


# ============================================================
# 5. Plot functions
# ============================================================

def plot_regression_clean(pred_df, output_path):
    pred_df = pred_df.copy()
    y_true = pred_df["y_true"].astype(float).values
    y_pred = pred_df["y_pred"].astype(float).values
    metrics = regression_metrics(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(y_true, y_pred, s=80, alpha=0.85)

    min_v = min(y_true.min(), y_pred.min())
    max_v = max(y_true.max(), y_pred.max())
    padding = (max_v - min_v) * 0.08 if max_v > min_v else 1.0
    min_v -= padding
    max_v += padding

    ax.plot([min_v, max_v], [min_v, max_v], linestyle="--", linewidth=2, color="black")
    ax.set_xlim(min_v, max_v)
    ax.set_ylim(min_v, max_v)
    ax.set_xlabel("Measured value")
    ax.set_ylabel("Predicted value")
    ax.text(
        0.05,
        0.95,
        f"R² = {metrics['r2']:.3f}\nRMSE = {metrics['rmse']:.3f}\nRPD = {metrics['rpd']:.3f}",
        transform=ax.transAxes,
        va="top",
        fontsize=20,
    )
    ax.tick_params(direction="out", length=6, width=1.5)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return metrics


def plot_residual_clean(pred_df, output_path):
    pred_df = pred_df.copy()
    pred_df["residual"] = pred_df["y_pred"] - pred_df["y_true"]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axhline(0, linestyle="--", linewidth=2, color="black")
    ax.scatter(pred_df["y_true"], pred_df["residual"], s=80, alpha=0.85)
    ax.set_xlabel("Measured value")
    ax.set_ylabel("Prediction residual")
    ax.tick_params(direction="out", length=6, width=1.5)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_combined_original_validation(original_df, validation_df, output_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    for ax, pred_df in zip(axes, [original_df, validation_df]):
        y_true = pred_df["y_true"].astype(float).values
        y_pred = pred_df["y_pred"].astype(float).values
        metrics = regression_metrics(y_true, y_pred)

        ax.scatter(y_true, y_pred, s=70, alpha=0.85)
        min_v = min(y_true.min(), y_pred.min())
        max_v = max(y_true.max(), y_pred.max())
        padding = (max_v - min_v) * 0.08 if max_v > min_v else 1.0
        min_v -= padding
        max_v += padding

        ax.plot([min_v, max_v], [min_v, max_v], linestyle="--", linewidth=2, color="black")
        ax.set_xlim(min_v, max_v)
        ax.set_ylim(min_v, max_v)
        ax.set_xlabel("Measured value")
        ax.set_ylabel("Predicted value")
        ax.text(
            0.05,
            0.95,
            f"R² = {metrics['r2']:.3f}\nRMSE = {metrics['rmse']:.3f}\nRPD = {metrics['rpd']:.3f}",
            transform=ax.transAxes,
            va="top",
            fontsize=20,
        )
        ax.tick_params(direction="out", length=6, width=1.5)
        for spine in ax.spines.values():
            spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def calculate_vip_from_plsr_pipeline(plsr_pipeline, wavelengths):
    if "reg" not in plsr_pipeline.named_steps:
        raise ValueError("Pipeline does not contain step named 'reg'.")

    pls = plsr_pipeline.named_steps["reg"]
    if not hasattr(pls, "x_scores_"):
        raise ValueError("The regression model does not have fitted PLS attributes.")

    T = pls.x_scores_
    W = pls.x_weights_
    Q = pls.y_loadings_
    p, h = W.shape

    ssy = np.sum(T ** 2, axis=0) * np.sum(Q ** 2, axis=0)
    total_ssy = np.sum(ssy)
    if total_ssy == 0:
        raise ValueError("Total explained Y variance is zero; VIP cannot be calculated.")

    W_norm = W / np.sqrt(np.sum(W ** 2, axis=0, keepdims=True))
    vip = np.sqrt(p * np.sum((W_norm ** 2) * ssy.reshape(1, -1), axis=1) / total_ssy)

    vip_df = pd.DataFrame({"wavelength": np.asarray(wavelengths, dtype=float), "VIP": vip})
    vip_df = vip_df.sort_values("VIP", ascending=False).reset_index(drop=True)
    return vip_df


def plot_top_vip(vip_df, output_path, top_n=10):
    top_df = vip_df.head(top_n).copy()
    top_df = top_df.sort_values("VIP", ascending=True)
    top_df["wavelength_label"] = top_df["wavelength"].round(2).astype(str)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top_df["wavelength_label"], top_df["VIP"])
    ax.axvline(1.0, linestyle="--", linewidth=2, color="black")
    ax.set_xlabel("VIP score")
    ax.set_ylabel("Wavelength (nm)")
    ax.tick_params(direction="out", length=6, width=1.5)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


# ============================================================
# 6. Load data
# ============================================================

print("Project root:")
print(PROJECT_DIR)
print("\n2024 patch CSV:")
print(A_PATCH_CSV)
print("\n2025 patch CSV:")
print(B_PATCH_CSV)
print("\nCombined 149-sample quality table:")
print(QUALITY_ALL_CSV)
print("\nProtein training split IDs:")
print(TRAIN_ID_CSV)
print("\nProtein validation split IDs:")
print(VAL_ID_CSV)

a_patch, a_wave_cols, a_wavelengths = prepare_patch_table(A_PATCH_CSV, "A_original")
b_patch, b_wave_cols, b_wavelengths = prepare_patch_table(B_PATCH_CSV, "B_second")

if len(a_wave_cols) != len(b_wave_cols):
    raise ValueError(
        f"A and B have different numbers of wavelength columns: A={len(a_wave_cols)}, B={len(b_wave_cols)}"
    )

a_waves = np.asarray(a_wavelengths, dtype=float)
b_waves = np.asarray(b_wavelengths, dtype=float)

if not np.allclose(a_waves, b_waves, atol=0.01):
    raise ValueError("A and B wavelength lists are not aligned. Please align/interpolate the spectra before regression.")

wavelengths = a_waves
wave_cols = a_wave_cols

# Rename B wavelength columns to A names if column formatting differs.
b_rename_wave = dict(zip(b_wave_cols, a_wave_cols))
b_patch = b_patch.rename(columns=b_rename_wave)

patch_all = pd.concat([a_patch, b_patch], axis=0, ignore_index=True)
quality_all = load_or_build_quality_all(a_patch_df=a_patch)

quality_all.to_csv(OUT_ROOT / "combined_A_B_quality_all149_used_for_six_regression.csv", index=False)

train_ids_protein, val_ids_protein = load_protein_split_ids()

print("\nPatch table:")
print(patch_all.shape)
print(patch_all["source"].value_counts())

print("\nQuality table:")
print(quality_all.shape)
print(quality_all["source"].value_counts())

print("\nDetected wavelengths:", len(wavelengths))
print("First 5:", wavelengths[:5])
print("Last 5:", wavelengths[-5:])


# ============================================================
# 7. Run one target using same protein split
# ============================================================

def run_one_target_regression_same_split(target_col):
    target_safe = safe_name(target_col)
    target_out = OUT_ROOT / target_safe
    target_out.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 100)
    print(f"Target: {target_col}")
    print("=" * 100)

    patch_target, quality_target = prepare_patch_for_target(patch_all, quality_all, target_col)

    split_quality_check = quality_target[
        quality_target["global_sample_id"].astype(str).isin(train_ids_protein + val_ids_protein)
    ].copy()

    missing_target_ids = split_quality_check.loc[
        split_quality_check["y_true"].isna(),
        "global_sample_id",
    ].astype(str).tolist()

    if len(missing_target_ids) > 0 and not DROP_MISSING_TARGET_IN_SAME_SPLIT:
        raise ValueError(
            f"Target {target_col} has missing values in the protein split.\n"
            f"Missing sample IDs:\n{missing_target_ids}\n\n"
            "Set DROP_MISSING_TARGET_IN_SAME_SPLIT = True if you want to keep the same split but drop missing-target samples."
        )

    train_patch = patch_target[patch_target["global_sample_id"].astype(str).isin(train_ids_protein)].copy()
    val_patch = patch_target[patch_target["global_sample_id"].astype(str).isin(val_ids_protein)].copy()

    if DROP_MISSING_TARGET_IN_SAME_SPLIT:
        train_patch = train_patch.dropna(subset=["y_true"]).copy()
        val_patch = val_patch.dropna(subset=["y_true"]).copy()

    train_sample_ids_actual = sorted(train_patch["global_sample_id"].astype(str).unique().tolist())
    val_sample_ids_actual = sorted(val_patch["global_sample_id"].astype(str).unique().tolist())

    if not DROP_MISSING_TARGET_IN_SAME_SPLIT:
        if sorted(train_ids_protein) != train_sample_ids_actual:
            missing_train = sorted(set(train_ids_protein) - set(train_sample_ids_actual))
            raise ValueError(f"Training samples do not match protein split. Missing: {missing_train}")
        if sorted(val_ids_protein) != val_sample_ids_actual:
            missing_val = sorted(set(val_ids_protein) - set(val_sample_ids_actual))
            raise ValueError(f"Validation samples do not match protein split. Missing: {missing_val}")

    print("Training samples:", train_patch["global_sample_id"].nunique())
    print("Validation samples:", val_patch["global_sample_id"].nunique())
    print("Training patches:", train_patch.shape[0])
    print("Validation patches:", val_patch.shape[0])

    print("\nTraining source counts:")
    print(train_patch.drop_duplicates("global_sample_id")["source"].value_counts())
    print("\nValidation source counts:")
    print(val_patch.drop_duplicates("global_sample_id")["source"].value_counts())

    train_patch.drop_duplicates("global_sample_id")[[
        "global_sample_id", "source", "sample_id", "treatment", "y_true"
    ]].to_csv(target_out / "training_samples_same_as_protein_split.csv", index=False)

    val_patch.drop_duplicates("global_sample_id")[[
        "global_sample_id", "source", "sample_id", "treatment", "y_true"
    ]].to_csv(target_out / "validation_samples_same_as_protein_split.csv", index=False)

    internal_metrics = []
    oof_prediction_tables = {}

    for model_name in MODEL_NAMES_TO_RUN:
        print(f"\nInternal GroupKFold model: {model_name}")
        try:
            m, oof_pred = groupkfold_oof_prediction(
                patch_train=train_patch,
                wave_cols=wave_cols,
                model_name=model_name,
                n_splits=5,
            )
            m["target"] = target_col
            internal_metrics.append(m)
            oof_prediction_tables[model_name] = oof_pred
            print(f"{model_name}: R²={m['r2']:.3f}, RMSE={m['rmse']:.3f}, MAE={m['mae']:.3f}, RPD={m['rpd']:.3f}")
        except Exception as e:
            print(f"Skipped {model_name} due to error: {e}")

    internal_summary = pd.DataFrame(internal_metrics)
    if internal_summary.empty:
        print(f"No valid internal models for target {target_col}.")
        return None

    internal_summary = internal_summary.sort_values(["r2", "rpd"], ascending=False).reset_index(drop=True)
    internal_summary.to_csv(target_out / "internal_GroupKFold_summary.csv", index=False)

    best_model_name = internal_summary.iloc[0]["model"]
    print("\nBest internal model:")
    print(best_model_name)
    display(internal_summary)

    for model_name, pred_df in oof_prediction_tables.items():
        pred_df.to_csv(target_out / f"OOF_predictions_{model_name}.csv", index=False)

    best_oof_pred = oof_prediction_tables[best_model_name].copy()

    final_model = model_factory(best_model_name)
    final_model, validation_pred = fit_predict_sample_level(
        model=final_model,
        train_patch_df=train_patch,
        test_patch_df=val_patch,
        wave_cols=wave_cols,
    )

    validation_metrics = regression_metrics(validation_pred["y_true"], validation_pred["y_pred"])
    validation_metrics["target"] = target_col
    validation_metrics["model"] = best_model_name
    validation_metrics["n_train_samples"] = train_patch["global_sample_id"].nunique()
    validation_metrics["n_validation_samples"] = validation_pred["global_sample_id"].nunique()
    validation_metrics["scenario"] = "same_fixed_split_as_protein"

    validation_metrics_df = pd.DataFrame([validation_metrics])
    validation_pred.to_csv(target_out / "fixed_validation_predictions_same_split_as_protein.csv", index=False)
    validation_metrics_df.to_csv(target_out / "fixed_validation_metrics_same_split_as_protein.csv", index=False)

    joblib.dump(
        {
            "target": target_col,
            "model_name": best_model_name,
            "model": final_model,
            "wavelengths": wavelengths,
            "wave_cols": wave_cols,
            "internal_summary": internal_summary,
            "train_ids_same_as_protein": train_ids_protein,
            "validation_ids_same_as_protein": val_ids_protein,
        },
        target_out / f"final_model_{best_model_name}_same_split_as_protein.joblib",
    )

    print("\nFixed validation performance:")
    display(validation_metrics_df)

    plot_regression_clean(best_oof_pred, target_out / "original_model_GroupKFold_measured_vs_predicted.png")
    plot_regression_clean(validation_pred, target_out / "fixed_validation_measured_vs_predicted_same_split_as_protein.png")
    plot_residual_clean(best_oof_pred, target_out / "original_model_GroupKFold_residual.png")
    plot_residual_clean(validation_pred, target_out / "fixed_validation_residual_same_split_as_protein.png")
    plot_combined_original_validation(
        best_oof_pred,
        validation_pred,
        target_out / "original_vs_validation_regression_plots_same_split_as_protein.png",
    )

    if "PLSR" in str(best_model_name).upper():
        try:
            vip_df = calculate_vip_from_plsr_pipeline(final_model, wavelengths)
            vip_df.to_csv(target_out / "VIP_scores_best_PLSR_model.csv", index=False)
            vip_df.head(10).to_csv(target_out / "Top10_VIP_wavelengths_best_PLSR_model.csv", index=False)
            plot_top_vip(vip_df, target_out / "Top10_VIP_wavelengths_best_PLSR_model.png", top_n=10)
            print("\nTop 10 VIP wavelengths:")
            display(vip_df.head(10))
        except Exception as e:
            print("VIP calculation failed:", e)

    return {
        "target": target_col,
        "best_model": best_model_name,
        "internal_best_r2": internal_summary.iloc[0]["r2"],
        "internal_best_rmse": internal_summary.iloc[0]["rmse"],
        "internal_best_mae": internal_summary.iloc[0]["mae"],
        "internal_best_rpd": internal_summary.iloc[0]["rpd"],
        "validation_r2": validation_metrics["r2"],
        "validation_rmse": validation_metrics["rmse"],
        "validation_mae": validation_metrics["mae"],
        "validation_bias": validation_metrics["bias"],
        "validation_rpd": validation_metrics["rpd"],
        "n_train_samples": train_patch["global_sample_id"].nunique(),
        "n_validation_samples": validation_pred["global_sample_id"].nunique(),
        "target_output_folder": str(target_out),
    }


# ============================================================
# 8. Run all six targets
# ============================================================

all_target_results = []

for target_col in TARGET_COLS:
    try:
        result = run_one_target_regression_same_split(target_col)
        if result is not None:
            all_target_results.append(result)
    except Exception as e:
        print("\n" + "!" * 100)
        print(f"Target failed: {target_col}")
        print("Error:", e)
        print("!" * 100)


overall_summary = pd.DataFrame(all_target_results)
overall_summary_path = OUT_ROOT / "six_indicator_regression_overall_summary_same_split_as_protein.csv"
overall_summary.to_csv(overall_summary_path, index=False)

print("\n" + "=" * 100)
print("Overall summary across six non-protein targets")
print("Same fixed split as protein regression")
print("=" * 100)

display(overall_summary)

print("\nSaved overall summary:")
print(overall_summary_path)
print("\nAll outputs saved to:")
print(OUT_ROOT)
